# 11 · Superficie construida con NDBI

**Objetivo:** Comparar intensidad urbana entre dos años.

**Datos:** Sentinel-2 SR: B11 y B8.

**Relevancia para política ambiental y social:** Apoya planificación urbana y evaluación de presión sobre ecosistemas.

**Limitaciones:** Suelo desnudo puede confundirse con áreas construidas.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
def mask_s2_sr(image):
    scl = image.select("SCL")
    clear = (
        scl.neq(3)   # sombra
        .And(scl.neq(8))  # nube media
        .And(scl.neq(9))  # nube alta
        .And(scl.neq(10)) # cirrus
        .And(scl.neq(11)) # nieve/hielo
    )
    return image.updateMask(clear).divide(10000).copyProperties(
        image, ["system:time_start"]
    )

def s2_composite(start, end):
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_sr)
        .median()
        .clip(aoi)
    )


In [ ]:
def ndbi(img):
    return img.normalizedDifference(["B11","B8"])
old = ndbi(s2_composite("2019-01-01","2019-12-31"))
new = ndbi(s2_composite("2025-01-01","2025-12-31"))
growth = new.subtract(old).rename("NDBI_change")
Map.addLayer(growth, {"min":-0.3,"max":0.3,"palette":["green","white","magenta"]}, "Cambio NDBI")
Map
